# Multiple Linear Regression

In real-world machine learning tasks, predictions are rarely based on a single input feature. **Multiple Linear Regression (MLR)** is the extension of Simple Linear Regression that allows us to model the relationship between multiple independent variables (features) and a single continuous dependent variable (target).

---

## 1. Geometric Intuition

How we visualize the regression model depends directly on the number of input features:

| Dimensions | Inputs | Output | Geometric Structure | Visualization |
| :--- | :--- | :--- | :--- | :--- |
| **2D Space** | 1 Feature ($X_1$) | 1 Target ($Y$) | **Line** | A 2D straight line ($Y = mX + c$) |
| **3D Space** | 2 Features ($X_1, X_2$) | 1 Target ($Y$) | **Plane** | A 2D flat surface floating in a 3D space |
| **Higher dimensions** | $m$ Features ($m \ge 3$) | 1 Target ($Y$) | **Hyperplane** | An $(m)$-dimensional mathematical boundary (cannot be visualized) |

> 📌 **Key Intuition:** No matter the number of dimensions, the core goal remains identical: find the linear structure (line, plane, or hyperplane) that minimizes the overall distance to all data points.

---


## 2. Mathematical Formulation

The equation for a hyperplane with $m$ features is written as:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_m X_m + \epsilon$$

Where:
- **$Y$**: The target variable.
- **$\beta_0$**: The Y-intercept (bias term).
- **$\beta_1, \beta_2, \dots, \beta_m$**: Coefficients (weights) for each independent feature.
- **$\epsilon$**: The stochastic error term (representing random variations not captured by features).

---

## 2.1 Representation using Matrices (Vectorization)

To solve the equation efficiently for a dataset containing $n$ observations and $m$ input features, we represent the data in matrix form.

### 1. The Input Matrix ($X$)
An $n \times (m+1)$ matrix. We add a column of $1$s at index 0 to account for the intercept term $\beta_0$:

$$X = \begin{bmatrix} 
1 & x_{11} & x_{12} & \dots & x_{1m} \\ 
1 & x_{21} & x_{22} & \dots & x_{2m} \\ 
\vdots & \vdots & \vdots & \ddots & \vdots \\ 
1 & x_{n1} & x_{n2} & \dots & x_{nm} 
\end{bmatrix}$$

### 2. The Coefficient Vector ($\beta$)
An $(m+1) \times 1$ column vector containing all weights and the bias:

$$\beta = \begin{bmatrix} \beta_0 \\ \beta_1 \\ \beta_2 \\ \vdots \\ \beta_m \end{bmatrix}$$

### 3. The Output Vector ($Y$)
An $n \times 1$ column vector containing actual values:

$$Y = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{bmatrix}$$

### The Prediction Formula
Using matrix multiplication, the predicted outputs $\hat{Y}$ can be computed as:

$$\hat{Y} = X \beta$$

---

## 2.2 Deriving the Closed-Form Solution (Ordinary Least Squares - OLS)

The loss function is the **Sum of Squared Residuals (SSR)**, which represents the squared error. In matrix form, this is:

$$E(\beta) = (Y - X\beta)^T(Y - X\beta)$$

Expanding this matrix dot product:

$$E(\beta) = Y^TY - Y^TX\beta - \beta^TX^TY + \beta^TX^TX\beta$$

Since $Y^TX\beta$ is a scalar ($1 \times 1$ matrix), it is equal to its transpose $\beta^TX^TY$. Therefore, we can group them:

$$E(\beta) = Y^TY - 2\beta^TX^TY + \beta^TX^TX\beta$$

To find the coefficients that minimize the error, we differentiate the loss function with respect to the vector $\beta$:

$$\frac{\partial E}{\partial \beta} = -2X^TY + 2X^TX\beta$$

Setting the derivative to zero to find the minimum:

$$-2X^TY + 2X^TX\beta = 0$$

$$X^TX\beta = X^TY$$

Solving for $\beta$ gives us the **OLS Closed-Form Solution**:

$$\beta = (X^TX)^{-1}X^TY$$

---

## 2.3 The Necessity of Gradient Descent

While the OLS equation allows us to find the optimal weights directly without training loops, it has a significant computational drawback:
- Calculating $\beta$ requires inverting the matrix $(X^TX)$, which is of size $(m+1) \times (m+1)$.
- Matrix inversion has a time complexity of **$\mathcal{O}(m^3)$**, where $m$ is the number of features.

### Comparison:

| Feature Count ($m$) | Matrix Size | OLS Feasibility | Alternative |
| :--- | :--- | :--- | :--- |
| **Small to Moderate** ($m < 10,000$) | Small | **Highly Feasible** (Calculates instantly) | None needed |
| **Very Large** ($m > 10,000$) | Large | **Computational Bottleneck** (Too slow/memory-intensive) | **Gradient Descent** (Iterative approximation technique) |

> 💡 **Takeaway:** For high-dimensional datasets (e.g., text processing, genomics), we bypass OLS and train linear models using iterative optimization techniques like **Gradient Descent**.

## 3. Code Implementation: scikit-learn vs. Custom NumPy Class

We will now write python code to:
1. Load a real-world dataset (the scikit-learn Diabetes dataset).
2. Train a scikit-learn `LinearRegression` model to establish a baseline.
3. Build a custom `MyLR` class from scratch using **NumPy matrix multiplication and inversion**.
4. Verify that our custom class yields coefficients and performance identical to scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')

# Load diabetes dataset
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset features: {X_train.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")


In [ ]:
# Train scikit-learn's LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict and calculate score
y_pred_lr = lr.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)

print("="*50)
print("SCIKIT-LEARN MODEL RESULTS")
print("="*50)
print(f"R2 Score:    {r2_lr:.6f}")
print(f"Intercept:   {lr.intercept_:.6f}")
print(f"Coefficients:\n{lr.coef_}")
print("="*50)


In [ ]:
# Building the Custom Linear Regression class using NumPy
class MyLR:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
        # 1. Insert a column of 1s at index 0 of the input matrix X to account for the intercept
        X_new = np.insert(X_train, 0, 1, axis=1)
        
        # 2. Implement the formula: beta = (X^T * X)^(-1) * X^T * y
        # np.linalg.inv: computes matrix inverse
        # .dot(): matrix multiplication / dot product
        # .T: transpose
        beta = np.linalg.inv(np.dot(X_new.T, X_new)).dot(X_new.T).dot(y_train)
        
        # 3. Extract parameters
        # beta[0] is the intercept term
        self.intercept_ = beta[0]
        # beta[1:] are the coefficients for the independent variables
        self.coef_ = beta[1:]
        
    def predict(self, X_test):
        # Perform matrix multiplication: y_pred = X * coefs + intercept
        return np.dot(X_test, self.coef_) + self.intercept_


In [ ]:
# Instantiate and train custom model
my_lr = MyLR()
my_lr.fit(X_train, y_train)

# Predict and calculate score
y_pred_my = my_lr.predict(X_test)
r2_my = r2_score(y_test, y_pred_my)

print("="*50)
print("CUSTOM MODEL RESULTS (MyLR)")
print("="*50)
print(f"R2 Score:    {r2_my:.6f}")
print(f"Intercept:   {my_lr.intercept_:.6f}")
print(f"Coefficients:\n{my_lr.coef_}")
print("="*50)

# Verify mathematical equivalence
assert np.allclose(lr.intercept_, my_lr.intercept_)
assert np.allclose(lr.coef_, my_lr.coef_)
assert np.allclose(r2_lr, r2_my)
print("Success Verification: Custom model outputs are mathematically identical to scikit-learn!")


In [ ]:
# 4. Visualizing a 3D regression plane (Two features -> One target)
# Let's use two features: CGPA and IQ to predict Package (LPA)
np.random.seed(0)
n = 100
cgpa = np.random.uniform(5, 10, n)
iq = np.random.uniform(80, 140, n)
package = 1.5 * cgpa + 0.04 * iq - 5 + np.random.normal(0, 0.5, n)

X_toy = np.column_stack((cgpa, iq))

# Fit model
toy_model = MyLR()
toy_model.fit(X_toy, package)

# Generate grid values for plotting the plane
x_surf, y_surf = np.meshgrid(np.linspace(5, 10, 10), np.linspace(80, 140, 10))
grid_points = np.column_stack((x_surf.ravel(), y_surf.ravel()))
z_surf = (toy_model.predict(grid_points)).reshape(x_surf.shape)

# 3D plotting
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Plot training data points
ax.scatter(cgpa, iq, package, color='#3498db', alpha=0.8, edgecolors='w', s=40, label='Data Points')

# Plot the regression plane
surf = ax.plot_surface(x_surf, y_surf, z_surf, color='#e74c3c', alpha=0.3, edgecolor='none')

ax.set_title('Multiple Linear Regression: 3D Visualization', fontsize=14, fontweight='bold')
ax.set_xlabel('CGPA')
ax.set_ylabel('IQ')
ax.set_zlabel('Package (LPA)')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Summary & Key Takeaways

- **Generalization:** Multiple Linear Regression is the generic form of linear regression, fitting a hyperplane $Y = X\beta$ when dealing with multiple predictors.
- **Linear Algebra in ML:** Building algorithms from scratch shows how basic linear algebra—matrix transposes, multiplication, and inverses—is used to solve optimization problems directly.
- **Closed-Form Limitations:** The OLS closed-form equation $\beta = (X^TX)^{-1}X^Ty$ is computationally efficient for small datasets but fails on very high-dimensional data due to the $\mathcal{O}(m^3)$ complexity of matrix inversion. For larger feature sizes, iterative techniques like **Gradient Descent** are required.

---

> 💡 **Practice Challenge:** Try passing a single feature (Simple Linear Regression) to the custom `MyLR` class. You will notice it works perfectly, confirming that the matrix-based formulation is a generalized solution for any number of input variables!